# Fábrica de Móveis — Resolução Detalhada, Passo a Passo
## Programação Linear: do problema ao ótimo, por três caminhos diferentes

> **Como usar:** abra este notebook no [Google Colab](https://colab.research.google.com) (Arquivo → Fazer upload de notebook, ou arraste este `.ipynb`). Rode as células na ordem com `Shift+Enter`.

Este é o problema-âncora do curso — o mesmo que aparece na Aula 1 (Programação Linear) e na Aula 2 (Simplex). Aqui reunimos a resolução **completa e detalhada** em um único lugar, mostrando três caminhos diferentes até a mesma resposta:

1. **Método gráfico** — desenhando a região viável e as retas de iso-lucro;
2. **Algoritmo Simplex** — tableau por tableau, à mão (com código que reproduz cada iteração);
3. **Solver profissional (PuLP)** — o que se usa na prática.

Os três caminhos devem chegar **exatamente à mesma solução** — essa é a prova de que cada método está correto.

## 1. O problema

> Uma marcenaria produz **mesas** e **cadeiras**.
>
> - Cada **mesa** dá lucro de **R$ 80** e consome **4 m² de madeira** e **2 h de carpintaria**.
> - Cada **cadeira** dá lucro de **R$ 60** e consome **2 m² de madeira** e **3 h de carpintaria**.
> - Por semana, há **40 m² de madeira** e **30 h de carpintaria** disponíveis.
>
> **Pergunta:** quantas mesas e cadeiras produzir por semana para **maximizar o lucro**?

## 2. Modelagem passo a passo (a receita dos 5 passos)

**Passo 1 — Leitura:** dois produtos concorrendo por dois recursos limitados (madeira e carpintaria) — mix de produção clássico.

**Passo 2 — Variáveis de decisão (com unidade!):**
- `x` = mesas produzidas por semana
- `y` = cadeiras produzidas por semana

**Passo 3 — Função objetivo:**

$$\text{Max } Z = 80x + 60y \quad \text{(lucro semanal, em R\$)}$$

**Passo 4 — Restrições:**

$$4x + 2y \le 40 \quad \text{(madeira, em m}^2\text{)}$$
$$2x + 3y \le 30 \quad \text{(carpintaria, em horas)}$$
$$x \ge 0, \quad y \ge 0$$

**Passo 5 — Conferência de unidades:** [m²/mesa]·[mesas] + [m²/cadeira]·[cadeiras] = m² ✔ (madeira); [h/mesa]·[mesas] + [h/cadeira]·[cadeiras] = h ✔ (carpintaria).

---
## 3. Caminho 1 — Resolvendo pelo método gráfico

Com apenas 2 variáveis, dá para resolver **desenhando**:

1. Desenhe os eixos x (mesas) e y (cadeiras) — só o 1º quadrante;
2. Transforme cada restrição em reta (troque `≤` por `=`) e ache os interceptos:
   - Madeira: `4x + 2y = 40` → corta os eixos em (10, 0) e (0, 20)
   - Carpintaria: `2x + 3y = 30` → corta os eixos em (15, 0) e (0, 10)
3. Identifique o lado válido de cada reta (teste a origem: 0 ≤ 40 ✔) e sombreie a **região viável**;
4. Desenhe retas de **iso-lucro** (`80x + 60y = Z`) — são paralelas; empurre-as na direção de crescimento de Z;
5. O último ponto de contato com a região viável é a **solução ótima**.

Vamos visualizar cada passo no código abaixo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Passo 2 e 3: desenhando a região viável ---
x = np.linspace(0, 12, 400)
y_madeira = (40 - 4*x) / 2       # 4x + 2y = 40  ->  y = (40 - 4x)/2
y_carpint = (30 - 2*x) / 3       # 2x + 3y = 30  ->  y = (30 - 2x)/3

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x, y_madeira, color='#2a78d6', linewidth=2, label='Madeira: 4x + 2y = 40')
ax.plot(x, y_carpint, color='#1baf7a', linewidth=2, label='Carpintaria: 2x + 3y = 30')

# A região viável fica ABAIXO das duas retas (e no 1º quadrante)
y_regiao = np.minimum(y_madeira, y_carpint)
ax.fill_between(x, 0, np.clip(y_regiao, 0, None), where=(y_regiao >= 0),
                color='#2a78d6', alpha=0.12, label='Região viável')

ax.set_xlim(0, 12); ax.set_ylim(0, 14)
ax.set_xlabel('x = mesas/semana'); ax.set_ylabel('y = cadeiras/semana')
ax.set_title('Passo 3: região viável — fábrica de móveis')
ax.grid(True, linewidth=0.5, alpha=0.4)
ax.legend()
plt.show()

In [ ]:
# --- Passo 4 e 5: retas de iso-lucro deslizando até o vértice ótimo ---
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x, y_madeira, color='#2a78d6', linewidth=2, label='Madeira')
ax.plot(x, y_carpint, color='#1baf7a', linewidth=2, label='Carpintaria')
ax.fill_between(x, 0, np.clip(np.minimum(y_madeira, y_carpint), 0, None),
                where=(np.minimum(y_madeira, y_carpint) >= 0), color='#2a78d6', alpha=0.12)

for Z in [300, 600, 900]:
    ax.plot(x, (Z - 80*x) / 60, linestyle='--', color='#898781', linewidth=1)
    ax.annotate(f'Z = {Z}', xy=(Z/80 - 1.3, 1.2), fontsize=9, color='#52514e')

# Ótimo: interseção das duas restrições
ax.scatter([7.5], [5], s=90, color='#0b0b0b', zorder=5)
ax.annotate('Ótimo (7,5 ; 5) — Z = R$ 900', xy=(7.5, 5),
            xytext=(8, 7.5), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='#0b0b0b'))

ax.set_xlim(0, 12); ax.set_ylim(0, 14)
ax.set_xlabel('x = mesas/semana'); ax.set_ylabel('y = cadeiras/semana')
ax.set_title('Passo 4-5: iso-lucros deslizando até o último vértice')
ax.grid(True, linewidth=0.5, alpha=0.4)
ax.legend()
plt.show()

### Conferindo pelo teorema fundamental

> Se um problema de PL tem solução ótima, **existe um vértice da região viável que é ótimo**.

Basta enumerar os vértices e calcular Z em cada um:

In [ ]:
vertices = {
    'Origem (0,0)':                  (0.0, 0.0),
    'Madeira ∩ eixo x (10,0)':       (10.0, 0.0),
    'Carpintaria ∩ eixo y (0,10)':   (0.0, 10.0),
    'Madeira ∩ Carpintaria (7.5,5)': (7.5, 5.0),
}

print(f"{'Vértice':<35} {'Z = 80x + 60y':>15}")
print('-' * 52)
melhor = max(vertices.items(), key=lambda kv: 80*kv[1][0] + 60*kv[1][1])
for nome, (vx, vy) in vertices.items():
    Z = 80*vx + 60*vy
    marca = '  <-- ÓTIMO' if nome == melhor[0] else ''
    print(f'{nome:<35} R$ {Z:>10.2f}{marca}')

**Resultado do Caminho 1 (gráfico): x = 7,5 · y = 5 · Z = R\$ 900.**

---
## 4. Caminho 2 — Resolvendo pelo Algoritmo Simplex (tableau)

O método gráfico só funciona com 2 variáveis. O **Simplex** resolve em qualquer dimensão, seguindo exatamente a mesma lógica: caminhar de vértice em vértice, sempre melhorando o lucro.

### 4.1 Forma padrão

Cada restrição `≤` ganha uma **variável de folga** (o quanto do recurso sobra):

$$\text{Max } Z = 80x + 60y + 0\!\cdot\!s_1 + 0\!\cdot\!s_2$$
$$4x + 2y + s_1 = 40 \quad \text{(madeira)}$$
$$2x + 3y + s_2 = 30 \quad \text{(carpintaria)}$$
$$x, y, s_1, s_2 \ge 0$$

### 4.2 Tableau 0 — ponto de partida (base = {s1, s2}, vértice (0,0), Z=0)

| Base | x | y | s1 | s2 | b |
|---|---|---|---|---|---|
| s1 | 4 | 2 | 1 | 0 | 40 |
| s2 | 2 | 3 | 0 | 1 | 30 |
| **Z** | **−80** | **−60** | 0 | 0 | **0** |

**Quem entra?** O coeficiente mais negativo da linha Z → **x** (cada mesa soma R\$ 80).

**Quem sai? (teste da razão)** `b ÷ coeficiente` em cada linha: s1 → 40÷4 = **10** (menor); s2 → 30÷2 = 15. A madeira estoura primeiro → **sai s1**, pivô = 4.

### 4.3 Tableau 1 — primeira parada (base = {x, s2}, vértice (10,0), Z=800)

| Base | x | y | s1 | s2 | b |
|---|---|---|---|---|---|
| x | 1 | 0,5 | 0,25 | 0 | 10 |
| s2 | 0 | 2 | −0,5 | 1 | 10 |
| **Z** | 0 | **−20** | 20 | 0 | **800** |

**Entra y** (cada cadeira ainda soma R\$ 20). **Teste da razão:** 10÷0,5 = 20; 10÷2 = **5** (menor) → **sai s2**, pivô = 2.

### 4.4 Tableau 2 — final (base = {x, y}, vértice (7,5 ; 5), Z=900)

| Base | x | y | s1 | s2 | b |
|---|---|---|---|---|---|
| x | 1 | 0 | 0,375 | −0,25 | 7,5 |
| y | 0 | 1 | −0,25 | 0,5 | 5 |
| **Z** | 0 | 0 | **15** | **10** | **900** |

**Critério de parada:** nenhum coeficiente negativo na linha Z → **ótimo!**

- **x = 7,5 mesas; y = 5 cadeiras; Z = R\$ 900** — o mesmo vértice do método gráfico;
- **Folgas** s1 = s2 = 0: os dois recursos foram totalmente consumidos (restrições **ativas**);
- Os números **15** e **10** são os **preços-sombra**: cada m² extra de madeira valeria R\$ 15 de lucro; cada hora extra de carpintaria, R\$ 10.

### 4.5 Vendo o Simplex rodar sozinho

A função abaixo reproduz **exatamente** os três tableaus acima, iteração por iteração — confira se os números batem com o que descrevemos em texto.

In [ ]:
import pandas as pd

def simplex_didatico(c, A, b, nomes_vars=None):
    """Resolve Max c.x s.a. A.x <= b, x >= 0, imprimindo cada tableau."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    c = np.array(c, dtype=float)
    m, n = A.shape
    nomes = (list(nomes_vars) if nomes_vars else [f'x{j+1}' for j in range(n)])
    nomes = nomes + [f's{i+1}' for i in range(m)]

    # ---- forma padrão: [A | I | b] e linha Z = [-c | 0 | 0] ----
    T = np.zeros((m + 1, n + m + 1))
    T[:m, :n] = A
    T[:m, n:n + m] = np.eye(m)
    T[:m, -1] = b
    T[-1, :n] = -c
    base = list(range(n, n + m))   # começa com as folgas na base (origem)

    it = 0
    while True:
        rotulos = [nomes[j] for j in base] + ['Z']
        df = pd.DataFrame(np.round(T, 3), columns=nomes + ['b'], index=rotulos)
        print(f'================ TABLEAU {it} ================')
        print(df.to_string(), '\n')

        # ---- quem entra? coluna mais negativa da linha Z ----
        col = int(np.argmin(T[-1, :-1]))
        if T[-1, col] >= -1e-9:
            print('Linha Z sem coeficientes negativos -> OTIMO encontrado!\n')
            break

        # ---- quem sai? teste da razão ----
        positivos = T[:m, col] > 1e-9
        if not positivos.any():
            print(f'Coluna de {nomes[col]} sem coeficiente positivo -> solucao ILIMITADA!')
            return None
        razoes = np.full(m, np.inf)
        razoes[positivos] = T[:m, -1][positivos] / T[:m, col][positivos]
        row = int(np.argmin(razoes))

        print(f'>>> Entra na base: {nomes[col]} (coef. {T[-1, col]:+.2f} na linha Z)')
        teste = {nomes[base[i]]: (round(razoes[i], 2) if np.isfinite(razoes[i]) else '—') for i in range(m)}
        print(f'>>> Teste da razao: {teste}')
        print(f'>>> Sai da base: {nomes[base[row]]} (menor razao = {razoes[row]:.2f})\n')

        # ---- pivoteamento (eliminação de Gauss) ----
        T[row] = T[row] / T[row, col]
        for i in range(m + 1):
            if i != row:
                T[i] = T[i] - T[i, col] * T[row]
        base[row] = col
        it += 1

    solucao = {nomes[j]: 0.0 for j in range(n)}
    for i, j in enumerate(base):
        if j < n:
            solucao[nomes[j]] = round(float(T[i, -1]), 6)
    print('Solucao otima :', solucao)
    print('Z otimo       :', round(float(T[-1, -1]), 6))
    sombra = {f'recurso {i+1} ({nomes[n+i]})': round(float(T[-1, n + i]), 4) for i in range(m)}
    print('Precos-sombra :', sombra)
    return solucao, float(T[-1, -1])

In [ ]:
simplex_didatico(
    c=[80, 60],
    A=[[4, 2],
       [2, 3]],
    b=[40, 30],
    nomes_vars=['mesas', 'cadeiras'],
)
# Compare a saida acima com os Tableaus 0, 1 e 2 descritos em texto na secao 4.2-4.4

**Resultado do Caminho 2 (Simplex): x = 7,5 · y = 5 · Z = R\$ 900 · preços-sombra 15 e 10.**

---
## 5. Caminho 3 — Confirmando com um solver profissional (PuLP)

Na prática, ninguém monta o tableau à mão — um **solver** faz isso (com muito mais eficiência, para problemas com milhares de variáveis). Por baixo dos panos, o solver do PuLP também executa uma variante do Simplex.

In [ ]:
# No Google Colab, o PuLP não vem pré-instalado — instale com o comando abaixo (uma vez por sessão)
!pip install pulp -q

In [ ]:
from pulp import LpMaximize, LpProblem, LpVariable, LpStatus, value

modelo = LpProblem('fabrica_de_moveis', LpMaximize)

x_var = LpVariable('mesas', lowBound=0)
y_var = LpVariable('cadeiras', lowBound=0)

modelo += 80*x_var + 60*y_var, 'lucro_total'          # Passo 3: função objetivo
modelo += 4*x_var + 2*y_var <= 40, 'madeira'           # Passo 4: restrição 1
modelo += 2*x_var + 3*y_var <= 30, 'carpintaria'       # Passo 4: restrição 2

modelo.solve()

print('Status  :', LpStatus[modelo.status])
print('Mesas   :', x_var.value())
print('Cadeiras:', y_var.value())
print('Lucro   : R$', value(modelo.objective))
print()
for nome, restricao in modelo.constraints.items():
    print(f'Preço-sombra de {nome}: {restricao.pi}  |  folga: {-restricao.slack:.2f}')

**Resultado do Caminho 3 (PuLP): x = 7,5 · y = 5 · Z = R\$ 900 · preços-sombra 15 e 10.**

---
## 6. Conferência cruzada — os três caminhos batem?

| Caminho | x (mesas) | y (cadeiras) | Z (lucro) | Preços-sombra |
|---|---|---|---|---|
| 1. Método gráfico | 7,5 | 5 | R$ 900 | — (não calculado diretamente) |
| 2. Simplex (tableau) | 7,5 | 5 | R$ 900 | madeira 15 · carpintaria 10 |
| 3. PuLP (solver) | 7,5 | 5 | R$ 900 | madeira 15 · carpintaria 10 |

Os três métodos concordam — essa é a prova de que a solução está correta, e a razão pela qual confiamos em um solver profissional para problemas grandes demais para desenhar ou tabelar à mão.

## 7. Leitura final

- **Decisão:** produzir **7,5 mesas** e **5 cadeiras** por semana, lucro máximo de **R\$ 900**;
- **Recursos:** madeira e carpintaria são consumidos **por completo** (folga zero nas duas restrições — ambas ficam **ativas**);
- **Preços-sombra:** 1 m² extra de madeira valeria R\$ 15 de lucro adicional; 1 hora extra de carpintaria valeria R\$ 10 — esse é o ponto de partida da **Análise de Sensibilidade**;
- **E o "7,5"?** Não dá para vender meia mesa — a Programação Linear assume que frações são aceitáveis (hipótese da divisibilidade). Esse é exatamente o gancho para a **Programação Inteira**.

**Desafio para praticar:** mude os números de `A`, `b` ou `c` nas células acima (por exemplo, o lucro da cadeira, ou a disponibilidade de madeira) e rode tudo de novo — os três caminhos devem continuar concordando entre si.

---

# ATIVIDADE PÃO

PF = 40l, 5k5, 2h  
PD = 50l, 4kg, 3h

disponivel = 100kg, 54h

In [2]:
from pulp import LpMaximize, LpProblem, LpVariable, LpStatus, value

# --- Interpretação dos dados para a 'ATIVIDADE PÃO' ---
# Assunções:
# - 'l' e 'kg' referem-se ao mesmo tipo de ingrediente.
# - 'h' refere-se a horas de trabalho.
# - Lucros: PF = R$ 40 e PD = R$ 50.
# - Consumo de ingrediente: PF = 4kg, PD = 5kg.

modelo_pao = LpProblem('atividade_pao', LpMaximize)

# Variáveis de decisão
x_pf = LpVariable('Pao_Frances', lowBound=0) # Quantidade de Pão Francês
x_pd = LpVariable('Pao_Doce', lowBound=0)    # Quantidade de Pão Doce

# Função objetivo: Maximizar o lucro total
# Lucro PF = 40, Lucro PD = 50
modelo_pao += 40 * x_pf + 50 * x_pd, 'lucro_total_pao'

# Restrições
# Restrição de Ingrediente (kg): 4kg para PF, 5kg para PD, 100kg disponíveis
modelo_pao += 4 * x_pf + 5 * x_pd <= 100, 'ingrediente_kg'

# Restrição de Horas de Trabalho (h): 2h para PF, 3h para PD, 54h disponíveis
modelo_pao += 2 * x_pf + 3 * x_pd <= 54, 'horas_trabalho'

# Resolver o modelo
modelo_pao.solve()

print('--- Resultado da Atividade Pão ---')
print('Status  :', LpStatus[modelo_pao.status])
print('Pão Francês (PF):', x_pf.value())
print('Pão Doce (PD)   :', x_pd.value())
print('Lucro Total     : R$', value(modelo_pao.objective))
print('\nPreços-sombra e Folgas:')
for nome, restricao in modelo_pao.constraints.items():
    print(f'  {nome}: Preço-sombra = {restricao.pi:.2f} | Folga = {-restricao.slack:.2f}')


--- Resultado da Atividade Pão ---
Status  : Optimal
Pão Francês (PF): 25.0
Pão Doce (PD)   : 0.0
Lucro Total     : R$ 1000.0

Preços-sombra e Folgas:
  ingrediente_kg: Preço-sombra = 10.00 | Folga = 0.00
  horas_trabalho: Preço-sombra = -0.00 | Folga = -4.00
